# STEP 1 — YouTube Ingestion Module 
Take a YouTube ad URL → validate → download → extract metadata → return structured output.

In [3]:
!pip install yt_dlp

  Using cached yt_dlp-2026.2.21-py3-none-any.whl.metadata (182 kB)
Using cached yt_dlp-2026.2.21-py3-none-any.whl (3.3 MB)


In [4]:
import os
import re
import json
from pathlib import Path
import yt_dlp

In [5]:
DOWNLOAD_DIR = "downloads"
Path(DOWNLOAD_DIR).mkdir(exist_ok=True)

# URL Validation Function

In [6]:
def validate_youtube_url(url: str) -> bool:
    youtube_regex = (
        r"(https?://)?(www\.)?"
        "(youtube\.com|youtu\.be)/.+"
    )
    return re.match(youtube_regex, url) is not None

<>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
/var/folders/sj/gg2grkd107n2cv3c8zq0ppvm0000gn/T/ipykernel_22223/2596461063.py:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  "(youtube\.com|youtu\.be)/.+"


In [7]:
validate_youtube_url("https://youtube.com/watch?v=dQw4w9WgXcQ")

True

# Video Download Function

In [8]:
def download_youtube_video(url: str, download_dir: str = DOWNLOAD_DIR):
    if not validate_youtube_url(url):
        raise ValueError("Invalid YouTube URL")

    ydl_opts = {
        'format': 'mp4',
        'outtmpl': os.path.join(download_dir, '%(title)s.%(ext)s'),
        'quiet': True,
        'noplaylist': True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        video_path = ydl.prepare_filename(info)

    metadata = {
        "title": info.get("title"),
        "duration": info.get("duration"),
        "channel": info.get("uploader"),
        "description": info.get("description")
    }

    return {
        "video_path": video_path,
        "metadata": metadata
    }

In [9]:
youtube_url = "https://youtu.be/Bcpu-jqAL6w?si=68H-Zzg9ASar2qLs"

result = download_youtube_video(youtube_url)

print(json.dumps(result["metadata"], indent=2))
print("\nSaved to:", result["video_path"])

{                            
  "title": "WHY DO IT? | NIKE",
  "duration": 60,
  "channel": "Nike",
  "description": "Why risk it? Because you can. #JustDoIt"
}

Saved to: downloads/WHY DO IT？ ｜ NIKE.mp4


# STEP 2 — Audio Extraction + Whisper Transcription
Extract audio from downloaded MP4 →
Transcribe using Whisper →
Store structured timestamped JSON

In [10]:
!pip install faster-whisper 

  Using cached faster_whisper-1.2.1-py3-none-any.whl.metadata (16 kB)
  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached faster_whisper-1.2.1-py3-none-any.whl (1.1 MB)
   ━━━━━━━━━━━━━━

In [12]:
# brew install ffmpeg [ do in terminal ] for mac

In [12]:
!pip install ffmpeg-python

  Using cached ffmpeg_python-0.2.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached future-1.0.0-py3-none-any.whl.metadata (4.0 kB)
Using cached ffmpeg_python-0.2.0-py3-none-any.whl (25 kB)
Using cached future-1.0.0-py3-none-any.whl (491 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ffmpeg-python]0m [ffmpeg-python]


In [13]:
import os
import json
import ffmpeg
from faster_whisper import WhisperModel

/Users/abhijnandas/Desktop/Agentic ai ad video project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Extract Audio from MP4

In [14]:
def extract_audio(video_path, output_audio_path="audio.wav"):
    (
        ffmpeg
        .input(video_path)
        .output(output_audio_path, format='wav', acodec='pcm_s16le', ac=1, ar='16000')
        .overwrite_output()
        .run(quiet=True)
    )
    return output_audio_path

# Initialize Whisper Model

In [15]:
model_size = "base"  # tiny, base, small, medium
model = WhisperModel(model_size, device="cpu", compute_type="int8")

# Transcription Function

In [16]:
def transcribe_audio(audio_path):
    segments, info = model.transcribe(audio_path)

    transcript_data = []

    for segment in segments:
        transcript_data.append({
            "start": round(segment.start, 2),
            "end": round(segment.end, 2),
            "text": segment.text.strip()
        })

    return transcript_data

In [29]:
!pip install ffmpeg-python

  Using cached ffmpeg_python-0.2.0-py3-none-any.whl.metadata (1.7 kB)
Using cached ffmpeg_python-0.2.0-py3-none-any.whl (25 kB)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [26]:
import ffmpeg
print(ffmpeg)
print(ffmpeg.__file__)

<module 'ffmpeg' from '/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/ffmpeg/__init__.py'>
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/ffmpeg/__init__.py


In [17]:
import ffmpeg
print(dir(ffmpeg))

['Error', 'Stream', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_ffmpeg', '_filters', '_probe', '_run', '_utils', '_view', 'colorchannelmixer', 'compile', 'concat', 'crop', 'dag', 'drawbox', 'drawtext', 'filter', 'filter_', 'filter_multi_output', 'get_args', 'hflip', 'hue', 'input', 'merge_outputs', 'nodes', 'output', 'overlay', 'overwrite_output', 'probe', 'run', 'run_async', 'setpts', 'trim', 'unicode_literals', 'vflip', 'view', 'zoompan']


In [18]:
import ffmpeg

print("Has input?:", hasattr(ffmpeg, "input"))
print(dir(ffmpeg))

Has input?: True
['Error', 'Stream', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_ffmpeg', '_filters', '_probe', '_run', '_utils', '_view', 'colorchannelmixer', 'compile', 'concat', 'crop', 'dag', 'drawbox', 'drawtext', 'filter', 'filter_', 'filter_multi_output', 'get_args', 'hflip', 'hue', 'input', 'merge_outputs', 'nodes', 'output', 'overlay', 'overwrite_output', 'probe', 'run', 'run_async', 'setpts', 'trim', 'unicode_literals', 'vflip', 'view', 'zoompan']


In [ ]:
# !pip uninstall ffmpeg -y
# !pip uninstall ffmpeg-python -y

Found existing installation: ffmpeg 1.4
Uninstalling ffmpeg-1.4:
  Successfully uninstalled ffmpeg-1.4
Found existing installation: ffmpeg-python 0.2.0
Uninstalling ffmpeg-python-0.2.0:
  Successfully uninstalled ffmpeg-python-0.2.0


In [ ]:
video_path = result["video_path"]

# 1️ Extract Audio
audio_path = extract_audio(video_path)

# 2️ Transcribe
transcript = transcribe_audio(audio_path)

# 3️ Save JSON
with open("transcript.json", "w") as f:
    json.dump(transcript, f, indent=2)

print("Transcription Complete ✅")
print("First 3 segments:\n")
print(transcript[:3])

Transcription Complete ✅
First 3 segments:

[{'start': 0.0, 'end': 2.0, 'text': "Why'd you make it harder on yourself?"}, {'start': 6.0, 'end': 8.0, 'text': "Why'd you make it harder on yourself?"}, {'start': 13.0, 'end': 15.0, 'text': "Why'd you chance it?"}]


# STEP 3 — OCR Text Extraction from Video Frames
Extract frames every N seconds →
Detect on-screen text using OCR →
Remove duplicates →
Store structured JSON

In [20]:
!pip install opencv-python pytesseract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 11.5 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 10.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pytesseract] [opencv-python]


In [22]:
!brew install tesseract

⠋ JSON API formula.jws.json                          Downloading  31.9MB/-------
⠋ JSON API cask.jws.json                             Downloading  15.3MB/-------⠋ JSON API formula.jws.json                          Downloading  31.9MB/-------
⠋ JSON API cask.jws.json                             Downloading  15.3MB/-------⠙ JSON API formula.jws.json                          Downloading  31.9MB/-------
⠙ JSON API cask.jws.json                             Downloading  15.3MB/-------⠙ JSON API formula.jws.json                          Downloading  31.9MB/-------
⠙ JSON API cask.jws.json                             Downloading  15.3MB/-------⠚ JSON API formula.jws.json                          Downloading 208.9KB/-------
⠚ JSON API cask.jws.json                             Downloading 311.3KB/-------⠚ JSON API formula.jws.json                          Downloading   1.2MB/-------
⠚ JSON API cask.jws.json                             Downloading 782.3KB/-------⠞ JSON API formula.jws.json       

In [23]:
import pytesseract
print(pytesseract.get_tesseract_version())

5.5.2


In [24]:
import cv2
import pytesseract
import os
import json

objc[22223]: Class AVFFrameReceiver is implemented in both /Users/abhijnandas/Desktop/Agentic ai ad video project/.venv/lib/python3.14/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x10d7d83a8) and /Users/abhijnandas/Desktop/Agentic ai ad video project/.venv/lib/python3.14/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x1194d03a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[22223]: Class AVFAudioReceiver is implemented in both /Users/abhijnandas/Desktop/Agentic ai ad video project/.venv/lib/python3.14/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x10d7d83f8) and /Users/abhijnandas/Desktop/Agentic ai ad video project/.venv/lib/python3.14/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x1194d03f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


# Frame Extraction Function
We extract one frame every 3 seconds

In [25]:
def extract_frames(video_path, interval_seconds=3):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    frame_interval = int(fps * interval_seconds)
    
    frames = []
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_count % frame_interval == 0:
            timestamp = frame_count / fps
            frames.append((timestamp, frame))
        
        frame_count += 1
    
    cap.release()
    return frames

# OCR Extraction Function
We clean the text and ignore empty/noisy detection

In [26]:
def extract_text_from_frames(frames):
    ocr_results = []
    seen_texts = set()
    
    for timestamp, frame in frames:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        text = pytesseract.image_to_string(gray)
        cleaned_text = text.strip().replace("\n", " ")
        
        # Filter small/noisy text
        if len(cleaned_text) > 5 and cleaned_text not in seen_texts:
            seen_texts.add(cleaned_text)
            ocr_results.append({
                "timestamp": round(timestamp, 2),
                "text": cleaned_text
            })
    
    return ocr_results

In [27]:
video_path = "downloads/WHY DO IT？ ｜ NIKE.mp4"

# 1️⃣ Extract frames
frames = extract_frames(video_path, interval_seconds=3)

print(f"Total frames extracted: {len(frames)}")

# 2️⃣ OCR detection
ocr_results = extract_text_from_frames(frames)

# 3️⃣ Save JSON
with open("ocr_results.json", "w") as f:
    json.dump(ocr_results, f, indent=2)

print("OCR Extraction Complete ✅")
print(ocr_results[:5])

Total frames extracted: 21
OCR Extraction Complete ✅
[{'timestamp': 29.61, 'text': 'Ae AAS'}]


# STEP 4 — Merge Transcript + OCR

Combine spoken + visual + metadata into one structured compliance document.

# Load Transcript + OCR

In [28]:
import json

with open("transcript.json", "r") as f:
    transcript = json.load(f)

with open("ocr_results.json", "r") as f:
    ocr_results = json.load(f)

print("Transcript segments:", len(transcript))
print("OCR detections:", len(ocr_results))

Transcript segments: 13
OCR detections: 1


# Convert Transcript to Structured Text

In [29]:
def format_transcript(transcript):
    lines = []
    for segment in transcript:
        line = f"[{segment['start']}s - {segment['end']}s] {segment['text']}"
        lines.append(line)
    return "\n".join(lines)

# Convert OCR to Structured Text

In [30]:
def format_ocr(ocr_results):
    lines = []
    for item in ocr_results:
        line = f"[{item['timestamp']}s] VISUAL TEXT: {item['text']}"
        lines.append(line)
    return "\n".join(lines)

# Merge Everything

In [31]:
video_metadata = result["metadata"]  # from Step 1

transcript_text = format_transcript(transcript)
ocr_text = format_ocr(ocr_results)

unified_document = f"""
VIDEO TITLE: {video_metadata['title']}
CHANNEL: {video_metadata['channel']}
DURATION: {video_metadata['duration']} seconds

--- SPOKEN CONTENT ---
{transcript_text}

--- VISUAL CONTENT ---
{ocr_text}
"""

with open("video_document.txt", "w") as f:
    f.write(unified_document)

print("Unified document created ✅")

Unified document created ✅


# STEP 5 — Build PDF-Based RAG Knowledge Base
Now we turn our 2 compliance PDFs into a semantic rule engine.

In [32]:
!pip install pypdf sentence-transformers faiss-cpu

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 8.7 MB/s  0:00:01eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 16.3 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 10.7 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.4 MB/s  0:00:00
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 13.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 11.2 MB/s  0:00:01 eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [sentence-transformers]ence-transformers]


In [33]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load PDFs

In [34]:
def extract_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

pdf1_text = extract_pdf_text("1001a-influencer-guide-508_1.pdf")
pdf2_text = extract_pdf_text("youtube-ad-specs.pdf")

combined_rules_text = pdf1_text + "\n" + pdf2_text

print("PDF text extracted ✅")

PDF text extracted ✅


# Chunk the Rules

In [35]:
def chunk_text(text, chunk_size=400, overlap=100):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    
    return chunks

rule_chunks = chunk_text(combined_rules_text)

print("Total rule chunks:", len(rule_chunks))

Total rule chunks: 63


# Create Embeddings

In [36]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

rule_embeddings = embedding_model.encode(rule_chunks)

print("Embeddings created ✅")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 663.65it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings created ✅


# Build FAISS Index

In [37]:
dimension = rule_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(rule_embeddings))

print("FAISS index built ✅")

FAISS index built ✅


# Save Index (Important!)

In [38]:
faiss.write_index(index, "rules_index.faiss")

np.save("rule_chunks.npy", np.array(rule_chunks))

print("Knowledge base saved ✅")

Knowledge base saved ✅


# STEP 6 — RAG Retrieval

Embed video document
Retrieve Top-K relevant compliance rule chunks
Prepare structured LLM input

# Load Saved Knowledge Base

In [39]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load FAISS index
index = faiss.read_index("rules_index.faiss")

# Load rule chunks
rule_chunks = np.load("rule_chunks.npy", allow_pickle=True)

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Knowledge base loaded ✅")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 733.00it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base loaded ✅


# Load Unified Video Document

In [40]:
with open("video_document.txt", "r") as f:
    video_document = f.read()

print("Video document length:", len(video_document))

Video document length: 672


# Embed Video Document

In [41]:
video_embedding = embedding_model.encode([video_document])
video_embedding = np.array(video_embedding)

print("Video embedding created ✅")

Video embedding created ✅


# Retrieve Top-K Relevant Rules

In [42]:
k = 5  # You can tune this later

distances, indices = index.search(video_embedding, k)

retrieved_rules = []

for idx in indices[0]:
    retrieved_rules.append(rule_chunks[idx])

print("Top-K rules retrieved ✅")
print("\n--- SAMPLE RULE ---\n")
print(retrieved_rules[0][:500])

Top-K rules retrieved ✅

--- SAMPLE RULE ---

0:20
– Consideration: 2:00 – 3:00
– Action: 0:15 – 0:20
Headline/Description ≤ 15 characters
Note: 
For optimal quality, do not use SD.
Audio  les are not accepted: MP3, WAV, or PCM.
Views count for YouTube only if the video exceeds 10 seconds.
Optimal consideration lift with ad formats between 60 to 180 seconds.
If you have a Call-to-action, the headline/description should be ≤ 10 characters.
Non


# STEP 7 — Groq Compliance Agent

In [43]:
!pip install groq

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 15.5 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [groq]6/7 [groq]tic]


In [ ]:
from groq import Groq
import os

GROQ_API_KEY = " i have removed the api key"  # Replace this
client = Groq(api_key=GROQ_API_KEY)

print("Groq client ready ✅")

Groq client ready ✅


# Prepare LLM Prompt

In [45]:
def build_prompt(video_text, retrieved_rules):
    rules_text = "\n\n".join(retrieved_rules)

    prompt = f"""
You are a regulatory compliance auditor.

Analyze the following video advertisement content.

VIDEO CONTENT:
{video_text}

RELEVANT COMPLIANCE RULES:
{rules_text}

Tasks:
1. Identify rule violations.
2. Quote exact evidence from video.
3. Assign severity: Low, Medium, High.
4. Assign confidence score between 0 and 1.
5. Return STRICT JSON in this format:

{{
  "violations": [
    {{
      "rule_reference": "",
      "evidence": "",
      "severity": "",
      "confidence": 0.0
    }}
  ]
}}

If no violations, return:
{{ "violations": [] }}
"""

    return prompt

# Call Groq LLM

Why temperature=0.2?
Lower temperature → more consistent structured JSON
Compliance systems need determinism.

In [ ]:
prompt = build_prompt(video_document, retrieved_rules)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.2 
)

llm_output = response.choices[0].message.content

print("LLM Response:\n")
print(llm_output)

LLM Response:

48 kHz, 24-bit
Video 1080p, 60fps
Video Title ≤ 15 characters
Video Description ≤ 200 words
Video Title ≤ 15 characters
Video Description ≤ 200 words
Video Title ≤ 15 characters
Video Description ≤ 200 words

The video content does not contain any explicit content, but it is a promotional video for Nike.

The video is 60 seconds long, and the questions posed in the video are meant to make the viewer question their motivation and willingness to take risks, which is a theme that Nike often uses in their advertising.

The video does not contain any gambling or lotteries, but it does contain the VISUAL TEXT: Ae AAS, which appears on screen at 29.61s.

The video is uploaded to the YouTube channel of Nike.

The video is in 1080p, 60fps, and the audio is 48 kHz, 24-bit.

The video title is "WHY DO IT? | NIKE" and the video description is not provided.

The video does not contain any explicit content, but it is a promotional video for Nike.

The video is 60 seconds long, and the

In [61]:
models = client.models.list()
for m in models.data:
    print(m.id)

openai/gpt-oss-20b
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi
moonshotai/kimi-k2-instruct
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-4-scout-17b-16e-instruct
whisper-large-v3-turbo
allam-2-7b
groq/compound
groq/compound-mini
meta-llama/llama-prompt-guard-2-22m
moonshotai/kimi-k2-instruct-0905
meta-llama/llama-guard-4-12b
openai/gpt-oss-safeguard-20b
qwen/qwen3-32b
llama-3.1-8b-instant
meta-llama/llama-4-maverick-17b-128e-instruct
llama-3.3-70b-versatile
whisper-large-v3
openai/gpt-oss-120b


In [66]:
# final
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a compliance auditor. Return ONLY valid JSON."},
        {"role": "user", "content": build_prompt(video_document[:2000], retrieved_rules[:3])}
    ],
    temperature=0,
    max_completion_tokens=800,
)

llm_output = response.choices[0].message.content
print(llm_output)

44.1 kHz, 16-bit
Video 1080p, 60fps


--- ANALYSIS ---
The video advertisement content does not contain any explicit claims or promises about the product. However, the spoken content seems to be questioning the viewer's decision-making and encouraging them to take risks. The visual content is minimal, with a brief appearance of the text "Ae AAS" which does not provide any clear meaning or relation to the product.


--- CONCLUSION ---
Based on the analysis, the video advertisement content appears to be more inspirational and motivational rather than promotional. It does not contain any explicit claims or promises about the product, and the visual content is minimal and does not provide any clear meaning or relation to the product.


--- RECOMMENDATIONS ---
The advertisement is likely to comply with the relevant compliance rules, as it does not contain any explicit claims or promises about the product. However, it is recommended to review the advertisement's audio and video quality to en

In [65]:
# testing 
# k=2
# response = client.chat.completions.create(
#     model="llama-3.1-8b-instant",
#     messages=[
#         {"role": "system", "content": "Return only JSON. No explanations."},
#         {"role": "user", "content": build_prompt(video_document[:1500], retrieved_rules[:2])}
#     ],
#     temperature=0,
#     max_completion_tokens=500
# )

# llm_output = response.choices[0].message.content
# print(llm_output)

# Parse JSON Safely

In [48]:
import json

try:
    violations_data = json.loads(llm_output)
    print("Parsed JSON successfully ✅")
except:
    print("⚠️ JSON parsing failed. Raw output below:")
    print(llm_output)

⚠️ JSON parsing failed. Raw output below:
48 kHz, 24-bit
Video 1080p, 60fps
Video Title ≤ 15 characters
Video Description ≤ 200 words
Video Title ≤ 15 characters
Video Description ≤ 200 words
Video Title ≤ 15 characters
Video Description ≤ 200 words

The video content does not contain any explicit content, but it is a promotional video for Nike.

The video is 60 seconds long, and the questions posed in the video are meant to make the viewer question their motivation and willingness to take risks, which is a theme that Nike often uses in their advertising.

The video does not contain any gambling or lotteries, but it does contain the VISUAL TEXT: Ae AAS, which appears on screen at 29.61s.

The video is uploaded to the YouTube channel of Nike.

The video is in 1080p, 60fps, and the audio is 48 kHz, 24-bit.

The video title is "WHY DO IT? | NIKE" and the video description is not provided.

The video does not contain any explicit content, but it is a promotional video for Nike.

The video 

# STEP 8 — Statistical Risk Engine

Convert severity → numeric
Combine with confidence
Compute overall risk score
Assign final risk category

In [49]:
severity_map = {
    "Low": 0.3,
    "Medium": 0.6,
    "High": 0.9
}

In [50]:
def compute_risk_score(violations_data):
    violations = violations_data.get("violations", [])
    
    if len(violations) == 0:
        return {
            "overall_risk_score": 0.0,
            "risk_level": "Low",
            "total_violations": 0
        }
    
    risk_values = []
    
    for v in violations:
        severity = v.get("severity", "Low")
        confidence = float(v.get("confidence", 0))
        
        severity_weight = severity_map.get(severity, 0.3)
        
        risk = severity_weight * confidence
        risk_values.append(risk)
    
    overall_risk = sum(risk_values) / len(risk_values)
    
    # Risk categorization
    if overall_risk < 0.3:
        risk_level = "Low"
    elif overall_risk < 0.7:
        risk_level = "Medium"
    else:
        risk_level = "High"
    
    return {
        "overall_risk_score": round(overall_risk, 3),
        "risk_level": risk_level,
        "total_violations": len(violations)
    }

In [52]:
print(llm_output)

48 kHz, 24-bit
Video 1080p, 60fps
Video Title ≤ 15 characters
Video Description ≤ 200 words
Video Title ≤ 15 characters
Video Description ≤ 200 words
Video Title ≤ 15 characters
Video Description ≤ 200 words

The video content does not contain any explicit content, but it is a promotional video for Nike.

The video is 60 seconds long, and the questions posed in the video are meant to make the viewer question their motivation and willingness to take risks, which is a theme that Nike often uses in their advertising.

The video does not contain any gambling or lotteries, but it does contain the VISUAL TEXT: Ae AAS, which appears on screen at 29.61s.

The video is uploaded to the YouTube channel of Nike.

The video is in 1080p, 60fps, and the audio is 48 kHz, 24-bit.

The video title is "WHY DO IT? | NIKE" and the video description is not provided.

The video does not contain any explicit content, but it is a promotional video for Nike.

The video is 60 seconds long, and the questions pose

In [53]:
import json

try:
    violations_data = json.loads(llm_output)
    print("Parsed JSON successfully ✅")
    print(violations_data)
except Exception as e:
    print("JSON parsing failed ❌")
    print("Error:", e)
    print("\nRaw LLM Output:\n")
    print(llm_output)

JSON parsing failed ❌
Error: Extra data: line 1 column 4 (char 3)

Raw LLM Output:

48 kHz, 24-bit
Video 1080p, 60fps
Video Title ≤ 15 characters
Video Description ≤ 200 words
Video Title ≤ 15 characters
Video Description ≤ 200 words
Video Title ≤ 15 characters
Video Description ≤ 200 words

The video content does not contain any explicit content, but it is a promotional video for Nike.

The video is 60 seconds long, and the questions posed in the video are meant to make the viewer question their motivation and willingness to take risks, which is a theme that Nike often uses in their advertising.

The video does not contain any gambling or lotteries, but it does contain the VISUAL TEXT: Ae AAS, which appears on screen at 29.61s.

The video is uploaded to the YouTube channel of Nike.

The video is in 1080p, 60fps, and the audio is 48 kHz, 24-bit.

The video title is "WHY DO IT? | NIKE" and the video description is not provided.

The video does not contain any explicit content, but it is

In [63]:
violations_data = extract_json(llm_output)
risk_summary = compute_risk_score(violations_data)

print(json.dumps(risk_summary, indent=2))

{
  "overall_risk_score": 0.0,
  "risk_level": "Low",
  "total_violations": 0
}


In [64]:
# risk_summary = compute_risk_score(violations_data)

# print("Risk Summary:\n")
# print(json.dumps(risk_summary, indent=2))

In [70]:
# !pip install gradio

In [ ]:
# import gradio as gr

# def compliance_pipeline(youtube_url):
#     try:
#         # Step 1: Download
#         result = download_youtube_video(youtube_url)
#         video_path = result["video_path"]

#         # Step 2: Extract Audio + Transcribe
#         audio_path = extract_audio(video_path)
#         transcript = transcribe_audio(audio_path)

#         # Step 3: OCR
#         frames = extract_frames(video_path, interval_seconds=3)
#         ocr_results = extract_text_from_frames(frames)

#         # Step 4: Merge
#         transcript_text = format_transcript(transcript)
#         ocr_text = format_ocr(ocr_results)

#         video_metadata = result["metadata"]

#         video_document = f"""
#         VIDEO TITLE: {video_metadata['title']}
#         CHANNEL: {video_metadata['channel']}
#         DURATION: {video_metadata['duration']} seconds

#         --- SPOKEN CONTENT ---
#         {transcript_text}

#         --- VISUAL CONTENT ---
#         {ocr_text}
#         """

#         # Step 5: RAG Retrieval
#         video_embedding = embedding_model.encode([video_document])
#         distances, indices = index.search(np.array(video_embedding), 3)
#         retrieved_rules = [rule_chunks[i] for i in indices[0]]

#         # Step 6: Groq LLM
#         response = client.chat.completions.create(
#             model="llama-3.3-70b-versatile",
#             messages=[
#                 {"role": "system", "content": "You are a compliance auditor. Return ONLY valid JSON."},
#                 {"role": "user", "content": build_prompt(video_document[:2000], retrieved_rules)}
#             ],
#             temperature=0,
#             max_completion_tokens=800,
#         )

#         llm_output = response.choices[0].message.content
#         violations_data = extract_json(llm_output)

#         # Step 7: Risk Engine
#         risk_summary = compute_risk_score(violations_data)

#         final_output = {
#             "violations": violations_data,
#             "risk_summary": risk_summary
#         }

#         return final_output

#     except Exception as e:
#         return {"error": str(e)}

In [71]:
# interface = gr.Interface(
#     fn=compliance_pipeline,
#     inputs=gr.Textbox(label="Paste YouTube Ad URL"),
#     outputs=gr.JSON(label="Compliance Report"),
#     title="Multimodal YouTube Ad Compliance Agent",
#     description="Paste a YouTube ad link. The system will analyze it using RAG + LLM and generate compliance risk assessment."
# )

# interface.launch()